# Setup and Configure NemoClaw on the Brev instance

This notebook is meant to run **on the Brev instance itself**. It shows how to use [`init_nemoclaw.sh`](init_nemoclaw.sh) to set up or reuse a **NemoClaw** sandbox, configure the **OpenShell** provider for NVIDIA Endpoints, copy **VSS skills** into the OpenClaw workspace, and update OpenClaw UI **`allowedOrigins`**.

**Default launchable:**  
[VSS + NemoClaw launchable](https://brev.nvidia.com/launchable/deploy/now?launchableID=env-3BgcwbtTMrB4IXdnaeDwaq5ULti)

If you are using that launchable, the VM is expected to already have the main prerequisites in place. If not, use the preflight cells below to confirm what is missing.

**What the script does now**

1. If the sandbox **already exists**, it **skips** NemoClaw onboard/install.
2. Otherwise it install NemoClaw or run NemoClaw onboard (non-interactive).
3. Configures the OpenShell provider for NVIDIA Endpoints.
4. Applies the VSS sandbox policy.
5. Uploads repo `skills/` into the sandbox workspace backing store (`/sandbox/.openclaw-data/workspace/skills`).
6. Updates OpenClaw UI `allowedOrigins` using [`update_openclaw_config.py`](update_openclaw_config.py).

**Security:** prefer `NVIDIA_API_KEY` from environment or Brev secrets. Do **not** commit notebook outputs that contain credentials.


## 1. Settings

<span style="color:red"><strong>Important:</strong> set <code>NVIDIA_API_KEY</code> here in via ENV. The NVIDIA_API_KEY is needed to run the steps below.</span>

In [ ]:
import os
from pathlib import Path

# ============= Set your NVIDIA API Key here (leave blank to use the existing environment variable)==============
NVIDIA_API_KEY = ""
# ===============================================================================================================


# No need to change these unless you want to customize the script.
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY") or NVIDIA_API_KEY
VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", "/home/ubuntu/video-search-and-summarization")).resolve()
NEMOCLAW_INIT_SCRIPT_PATH = VSS_REPO_DIR / "scripts" / "nemoclaw" / "init_nemoclaw.sh"
NEMOCLAW_INSTALL = Path("/home/ubuntu/NemoClaw/install.sh")
POLICY_PATH = VSS_REPO_DIR / "assets" / "vss_nemoclaw_policy.yaml"
UPDATE_CONFIG = VSS_REPO_DIR / "scripts" / "nemoclaw" / "update_openclaw_config.py"
SKILLS_DIR = VSS_REPO_DIR / "skills"
NEMOCLAW_SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", "demo")
LAUNCHABLE_URL = (
    "https://brev.nvidia.com/launchable/deploy/now"
    "?launchableID=env-3BgcwbtTMrB4IXdnaeDwaq5ULti"
)

print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("Launchable:", LAUNCHABLE_URL)
print("\nVSS_REPO_DIR:", VSS_REPO_DIR)
print("NEMOCLAW_INIT_SCRIPT_PATH:", NEMOCLAW_INIT_SCRIPT_PATH)
print("NEMOCLAW_INSTALL:", NEMOCLAW_INSTALL)
print("POLICY_PATH:", POLICY_PATH)
print("UPDATE_CONFIG:", UPDATE_CONFIG)
print("SKILLS_DIR:", SKILLS_DIR)

## 2. Preflight

Run the next cell to confirm the expected files and commands are present on the instance.


In [ ]:
import os
import shutil
from pathlib import Path

checks = {
    "init_nemoclaw.sh": NEMOCLAW_INIT_SCRIPT_PATH.is_file(),
    "NemoClaw install.sh": NEMOCLAW_INSTALL.is_file() and os.access(NEMOCLAW_INSTALL, os.X_OK),
    "vss_nemoclaw_policy.yaml": POLICY_PATH.is_file(),
    "update_openclaw_config.py": UPDATE_CONFIG.is_file(),
    "skills/": SKILLS_DIR.is_dir(),
    "docker": shutil.which("docker") is not None,
    "openshell": shutil.which("openshell") is not None,
    "python3": shutil.which("python3") is not None,
    "nemoclaw (optional)": shutil.which("nemoclaw") is not None,
}

for label, ok in checks.items():
    print(f"{'OK ' if ok else 'NO '} {label}")

## 3. Install and Configure NemoClaw (OpenClaw) for VSS skills

The next cell runs the script in the foreground so you can watch progress.

The script will:

- create or update the NemoClaw sandbox,
- configure the OpenShell provider,
- apply VSS NemoClaw policy,
- install VSS skills,
- update OpenClaw UI origin setting.

In [ ]:
import os
import subprocess
from pathlib import Path

init_path = Path(NEMOCLAW_INIT_SCRIPT_PATH)
if not init_path.is_file():
    raise FileNotFoundError(f"Missing init script: {init_path}")

env = os.environ.copy()
env["VSS_REPO_DIR"] = VSS_REPO_DIR
env["NEMOCLAW_SANDBOX_NAME"] = NEMOCLAW_SANDBOX_NAME
if NVIDIA_API_KEY.strip():
    env["NVIDIA_API_KEY"] = NVIDIA_API_KEY.strip()

timeout_sec = 3600

print("Running:", init_path, flush=True)
r = subprocess.run(
    ["bash", str(init_path)],
    env=env,
    cwd=str(init_path.parent),
    timeout=timeout_sec,
)
if r.returncode != 0:
    raise RuntimeError(f"init_nemoclaw.sh exited with {r.returncode}")
print("Done.", flush=True)


## 4. Verify sandbox, skills, `allowedOrigins`, and the UI

The next cell checks:

- whether the sandbox exists,
- the current active sandbox policy,
- whether VSS skills landed in `/sandbox/.openclaw-data/workspace/skills`,
- the OpenClaw config path used by this setup,
- the current `allowedOrigins` value.


In [ ]:
import json
import shutil
import subprocess

def run(cmd):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout)
    if r.stderr:
        print(r.stderr)
    return r

run(["openshell", "sandbox", "get", NEMOCLAW_SANDBOX_NAME])
run(["openshell", "policy", "get", NEMOCLAW_SANDBOX_NAME])

gateway_container = subprocess.run(
    ["docker", "ps", "--format", "{{.Names}}"], capture_output=True, text=True, check=True
).stdout.splitlines()
gateway_container = next((name for name in gateway_container if name.startswith("openshell-cluster-")), None)
print("Gateway container:", gateway_container)

## 5. Open the UI and smoke test it

### Test 1. Open the OpenClaw UI

After `init_nemoclaw.sh` finishes, look for the **OpenClaw UI link** in the script output. The setup should print a URL after `allowedOrigins` is configured.

Open that link in your browser.

![OpenClaw UI link screenshot](./images/OpenClawUILink.png)

### Test 2. Verify the LLM works in chat

Start a chat with the agent and send a simple prompt such as:

- `hello`
- `what model are you using?`
- `list your available skills`

![OpenClaw UI link screenshot](./images/OpenClawUIChat.png)

You should get a normal model response back. If the UI opens but chat fails, re-check provider setup and the active policy.

### Test 3. Verify the VSS skills are imported

Open the **Skills** tab in the UI and confirm the **VSS skills** are present.

![OpenClaw UI link screenshot](./images/OpenClawUISkills.png)

If the UI opens and chat works but the Skills tab is missing VSS skills, re-check `/sandbox/.openclaw-data/workspace/skills` in the verify step above.


## 6. Unintall NemoClaw ##

In [ ]:
import subprocess

cmd = "curl -fsSL https://raw.githubusercontent.com/NVIDIA/NemoClaw/refs/heads/main/uninstall.sh | bash -s -- --yes"
print(f"$ {cmd}")
subprocess.run(["bash", "-lc", cmd], check=True)